# Run Broadcasting Experiments

Use this notebook for the main protocol workflow: exact simulation, QEC Monte Carlo sampling, a single IBM hardware point, or an IBM hardware tau sweep. Results are saved through the unified JSON schema in `results/`.


In [7]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from broadcasting import (
    ExactBackend,
    HardwareBackend,
    ProtocolConfig,
    SamplingBackend,
    load_run,
    save_run,
)
from broadcasting.plotting import plot_fidelity_vs_noise, plot_run_sweep, save_figure

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120})


## Configuration

Set `MODE` to `exact`, `sampling`, `hardware`, or `hardware_tau_sweep`.


In [8]:
MODE = "exact"

M = 1
N = 2
alpha = 1.0 / np.sqrt(2)
use_qec = False
outcomes = [0] * M
seed = 42

rng = np.random.default_rng(seed)
nt = 1
theta_samples = rng.uniform(0, 2 * np.pi, size=(nt, M)).tolist()
thetas = theta_samples[0]

p_list = np.linspace(0, 1, 50).tolist()
n_samples = 1000

tau_values = np.linspace(0, 6000, 121).astype(int).tolist()
tau = tau_values[0]

IBM_PROFILE = "mprest1"
IBM_BACKEND = "ibm_brisbane"
OPTIMIZATION_LEVEL = 3
SHOTS = 4096

SAVE_FIGURES = False
FIGURE_DIR = Path("figures")

print(f"Mode={MODE}  M={M}  N={N}  QEC={use_qec}")
print("theta samples:")
for i, sample in enumerate(theta_samples):
    print(f"  {i}: {np.array(sample)}")


Mode=exact  M=1  N=2  QEC=False
theta samples:
  0: [4.86290927]


## Run


In [ ]:
if MODE == "sampling" and not use_qec:
    raise ValueError("MODE='sampling' requires use_qec=True.")

config = ProtocolConfig(
    M=M,
    N=N,
    alpha=alpha,
    thetas=thetas,
    p_list=p_list if MODE in {"exact", "sampling"} else [],
    use_qec=use_qec,
    outcomes_list=outcomes,
    tau=tau if MODE == "hardware" else None,
    n_samples=n_samples if MODE == "sampling" else None,
    seed=seed,
)

if MODE == "exact":
    backend = ExactBackend()
    result = backend.run(config)
elif MODE == "sampling":
    backend = SamplingBackend(n_samples=n_samples, seed=seed)
    result = backend.run(config)
elif MODE in {"hardware", "hardware_tau_sweep"}:
    from qiskit_ibm_runtime import QiskitRuntimeService

    service = QiskitRuntimeService(name=IBM_PROFILE)
    backend = HardwareBackend(
        service=service,
        backend_name=IBM_BACKEND,
        shots=SHOTS,
        optimization_level=OPTIMIZATION_LEVEL,
    )
    if MODE == "hardware":
        result = backend.run(config)
    else:
        result = backend.run_tau_sweep(config, tau_values, theta_samples=theta_samples)
else:
    raise ValueError(f"Unknown MODE: {MODE}")

print(f"Recorded mode: {result.metadata.get('mode')}")
print(f"Fidelity array shape: {np.asarray(result.fidelities).shape}")


## Save And Plot


In [ ]:
out_path = save_run(result, config)
saved_run = load_run(out_path)
print(f"Saved to {out_path}")

if saved_run["sweep"]["axis"] == "p":
    fids = np.asarray(saved_run["fidelities"], dtype=float)
    fig = plot_fidelity_vs_noise(
        np.asarray(saved_run["sweep"]["values"], dtype=float),
        {f"Receiver {i}": fids[:, i] for i in range(saved_run["N"])},
        mode_label=saved_run.get("backend", MODE),
        protocol_info={"M": M, "N": N, "use_qec": use_qec},
        show=False,
    )
else:
    fig = plot_run_sweep(
        saved_run,
        tau_scale=4e-3,
        tau_label="Idle delay (us)",
        show=False,
    )

plt.tight_layout()
if SAVE_FIGURES:
    figure_path = FIGURE_DIR / f"{Path(out_path).stem}.png"
    save_figure(fig, figure_path)
    print(f"Saved title-free figure to {figure_path}")
plt.show()


## Optional Exact Vs Sampling Overlay

`SamplingBackend` is for the QEC path, so this comparison runs only when `use_qec=True`.


In [ ]:
RUN_COMPARISON = False

if RUN_COMPARISON and use_qec:
    compare_config = ProtocolConfig(
        M=M,
        N=N,
        alpha=alpha,
        thetas=thetas,
        p_list=p_list,
        use_qec=True,
        outcomes_list=outcomes,
        seed=seed,
    )
    exact = ExactBackend().run(compare_config)
    sampled = SamplingBackend(n_samples=5000, seed=seed).run(compare_config)

    p_arr = np.asarray(p_list, dtype=float)
    exact_avg = np.asarray(exact.fidelities, dtype=float).mean(axis=1)
    sampled_avg = np.asarray(sampled.fidelities, dtype=float).mean(axis=1)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(p_arr, exact_avg, label="Exact")
    ax.plot(p_arr, sampled_avg, "--", label="Sampling")
    ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5)
    ax.set_xlabel("Depolarizing probability p")
    ax.set_ylabel("Average fidelity")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / "exact_vs_sampling.png")
    plt.show()
elif RUN_COMPARISON:
    print("Set use_qec=True before running the sampling comparison.")


## Optional Sampling Convergence


In [ ]:
RUN_CONVERGENCE = False

if RUN_CONVERGENCE:
    conv_config = ProtocolConfig(
        M=1,
        N=2,
        alpha=1.0 / np.sqrt(2),
        thetas=[0.0],
        p_list=np.linspace(0, 1, 21).tolist(),
        use_qec=True,
        outcomes_list=[0],
        seed=0,
    )
    exact_fids = np.asarray(ExactBackend().run(conv_config).fidelities)
    p_arr = np.asarray(conv_config.p_list)
    n_sweep = [50, 100, 200, 500, 1000, 2000, 5000, 10000]
    seeds = [0, 1, 2, 3, 4]

    # Multiple seeds so the fit isn't driven by one noisy trajectory realization.
    errors = np.zeros((len(seeds), len(n_sweep)))
    for si, seed in enumerate(seeds):
        for ni, ns in enumerate(n_sweep):
            sampled = SamplingBackend(n_samples=ns, seed=seed).run(conv_config)
            diff = np.abs(np.asarray(sampled.fidelities) - exact_fids)
            errors[si, ni] = np.trapz(diff, p_arr, axis=0).sum()
        print(f"seed={seed}: " + ", ".join(f"n={ns}:{errors[si, ni]:.4f}" for ni, ns in enumerate(n_sweep)))

    mean_errors = errors.mean(axis=0)
    std_errors = errors.std(axis=0)

    # Fit log(error) = slope*log(n) + intercept by linear regression, instead
    # of overlaying an assumed 1/sqrt(n) reference anchored to a single point --
    # this is what actually establishes the exponent rather than assuming it.
    log_n = np.log(n_sweep)
    log_err = np.log(mean_errors)
    slope, intercept = np.polyfit(log_n, log_err, 1)
    residuals = log_err - (slope * log_n + intercept)
    dof = len(n_sweep) - 2
    slope_se = (
        np.sqrt(np.sum(residuals ** 2) / dof / np.sum((log_n - log_n.mean()) ** 2))
        if dof > 0 else float("nan")
    )
    print(f"Fitted exponent: {slope:.3f} +/- {slope_se:.3f} (expect -0.5 for 1/sqrt(n) scaling)")

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.errorbar(
        n_sweep, mean_errors, yerr=std_errors, fmt="o", capsize=3,
        label=f"Observed ({len(seeds)} seeds, mean +/- std)",
    )
    fit_line = np.exp(intercept) * np.asarray(n_sweep, dtype=float) ** slope
    ax.plot(n_sweep, fit_line, "--", color="gray", label=f"Fit: n^{slope:.3f} +/- {slope_se:.3f}")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Number of samples")
    ax.set_ylabel("Error area")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend()
    plt.tight_layout()
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / "sampling_convergence.png")
        save_figure(fig, Path("manuscript") / "mc_sampling_convergence.png")
    plt.show()



## Optional: Structured Vs Generic Circuit Comparison (Preliminary Hardware Test)

Compares the existing generic circuit (`qc.initialize` + exponential feedforward) against the new
structured circuit (Dicke-state prep + linear feedforward) for the `M`, `N`, `alpha`, `thetas` set in
the **Configuration** cell above. Structured prep only supports `N in {1, 2}`.

The next cell is **local-only** (no IBM account needed): noiseless fidelity check for both circuits.
The cell after that **submits to hardware** and only runs if you set
`SUBMIT_STRUCTURED_COMPARISON = True` -- it uses whichever real backend is currently least busy.


In [9]:
from broadcasting.circuit import generate_qiskit_circuit
from broadcasting.fidelity import add_fidelity as add_fidelity_readout
from qiskit_aer import AerSimulator


def _build_variant(structured):
    qc = generate_qiskit_circuit(M, N, thetas, alphas=alpha, tau=0, use_structured_prep=structured)
    return add_fidelity_readout(qc, N=N, thetas=thetas, alpha=alpha)


print(f"M={M}, N={N}, alpha={alpha:.4f}, thetas={np.round(thetas, 4).tolist()}")
for label, structured in (("generic", False), ("structured", True)):
    qc, reg_name, _ = _build_variant(structured)

    counts = AerSimulator().run(qc, shots=4096, seed_simulator=0).result().get_counts()
    total = sum(counts.values())
    fids = [
        sum(c for bs, c in counts.items() if bs.split()[0][N - 1 - i] == "0") / total
        for i in range(N)
    ]
    print(f"  {label:10s}: qubits={qc.num_qubits}, depth={qc.depth()}, fidelities={[f'{f:.3f}' for f in fids]}")


M=1, N=2, alpha=0.7071, thetas=[4.8629]
  generic   : qubits=4, depth=9, fidelities=['1.000', '1.000']
  structured: qubits=4, depth=13, fidelities=['1.000', '1.000']


In [10]:
SUBMIT_STRUCTURED_COMPARISON = True  # set True to actually spend hardware time/shots

if SUBMIT_STRUCTURED_COMPARISON:
    from qiskit_ibm_runtime import QiskitRuntimeService

    service = QiskitRuntimeService(name=IBM_PROFILE)

    for structured in (False, True):
        compare_config = ProtocolConfig(
            M=M,
            N=N,
            alpha=alpha,
            thetas=thetas,
            outcomes_list=outcomes,
            tau=0,
            seed=seed,
            use_structured_prep=structured,
        )
        # backend_name=None -> whichever real backend is currently least busy.
        hw_backend = HardwareBackend(service=service, shots=SHOTS, optimization_level=OPTIMIZATION_LEVEL)
        result = hw_backend.run(compare_config)
        label = "structured" if structured else "generic"
        print(f"{label:10s}: backend={result.metadata['backend']} job={result.metadata['job_id']} fidelities={result.fidelities}")
        print(f"  saved to {save_run(result, compare_config)}")
else:
    print("SUBMIT_STRUCTURED_COMPARISON is False -- nothing sent to hardware.")


management.get:WARNING:2026-09-08 13:24:36,023: Loading saved account: mprest1


generic   : backend=ibm_marrakesh job=dag49mni3e6s738lteu0 fidelities=[0.909423828125, 0.821533203125]
  saved to results/run_20260908_132728_4fc6f552.json
structured: backend=ibm_marrakesh job=dag4b0hhvn6c73cq3k80 fidelities=[0.872314453125, 0.793701171875]
  saved to results/run_20260908_132738_ef85f542.json
